# 2.4 Convolutional Nerual Network Hyperparamter Tuning

## Table of Contents
### 1. Import Libraries and Data

## 1. Import Libraries and Data

In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import os
import operator
import time
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import make_scorer, accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.multiclass import type_of_target
import tensorflow as tf
from numpy import unique
from numpy import reshape
from tensorflow.keras.models import Sequential
from sklearn.model_selection import cross_val_score
from tensorflow.keras.layers import Input, Conv1D, Dense, Dropout, BatchNormalization, Flatten, MaxPooling1D
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam, SGD, RMSprop, Adadelta, Adagrad, Adamax, Nadam, Ftrl
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from scikeras.wrappers import KerasClassifier  # Use scikeras for scikit-learn compatibility
from math import floor
from bayes_opt import BayesianOptimization
from tensorflow.keras.layers import LeakyReLU  # Use tensorflow.keras instead of keras
LeakyReLU = LeakyReLU(negative_slope=0.1)
import warnings

In [9]:
# Import unscaled weather data
path = r'/Users/daxma/OneDrive/Desktop/Data Analytics/Machine Learning with Python/07-25 ClimateWins Analysis/02 Data'
climate = pd.read_csv(os.path.join(path, 'Prepared Data', 'weather_unscaled_clean.csv'))

# Check output
climate.head()

,BASEL_cloud_cover,BASEL_humidity,BASEL_pressure,BASEL_global_radiation,BASEL_precipitation,BASEL_sunshine,BASEL_temp_mean,BASEL_temp_min,BASEL_temp_max,BELGRADE_cloud_cover,...,STOCKHOLM_temp_max,VALENTIA_cloud_cover,VALENTIA_humidity,VALENTIA_pressure,VALENTIA_global_radiation,VALENTIA_precipitation,VALENTIA_sunshine,VALENTIA_temp_mean,VALENTIA_temp_min,VALENTIA_temp_max
0,7,0.85,1.018,0.32,0.09,0.7,6.5,0.8,10.9,1,...,4.9,5,0.88,1.0003,0.45,0.34,4.7,8.5,6.0,10.9
1,6,0.84,1.018,0.36,1.05,1.1,6.1,3.3,10.1,6,...,5.0,7,0.91,1.0007,0.25,0.84,0.7,8.9,5.6,12.1
2,8,0.90,1.018,0.18,0.30,0.0,8.5,5.1,9.9,6,...,4.1,7,0.91,1.0096,0.17,0.08,0.1,10.5,8.1,12.9
3,3,0.92,1.018,0.58,0.00,4.1,6.3,3.8,10.6,8,...,2.3,7,0.86,1.0184,0.13,0.98,0.0,7.4,7.3,10.6
4,6,0.95,1.018,0.65,0.14,5.4,3.0,-0.7,6.0,8,...,4.3,3,0.80,1.0328,0.46,0.00,5.7,5.7,3.0,8.4


In [11]:
# Import pleasant weather data
weather = pd.read_csv(os.path.join(path, 'Original Data', 'Pleasant_Weather_Answers.csv'))

# Check output 
weather.head()

,DATE,BASEL_pleasant_weather,BELGRADE_pleasant_weather,BUDAPEST_pleasant_weather,DEBILT_pleasant_weather,DUSSELDORF_pleasant_weather,HEATHROW_pleasant_weather,KASSEL_pleasant_weather,LJUBLJANA_pleasant_weather,MAASTRICHT_pleasant_weather,MADRID_pleasant_weather,MUNCHENB_pleasant_weather,OSLO_pleasant_weather,SONNBLICK_pleasant_weather,STOCKHOLM_pleasant_weather,VALENTIA_pleasant_weather
0,19600101,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,19600102,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,19600103,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,19600104,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,19600105,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 2. Data Cleaning

In [13]:
# Drop date from weather df
weather.drop(columns = 'DATE', inplace = True)

In [15]:
climate.shape

(22950, 135)

In [17]:
weather.shape

(22950, 15)

## 3. Data Reshaping

In [19]:
# Turn X and answers from df to arrays

X = np.array(climate)
y = np.array(weather)

In [21]:
X = X.reshape(-1,15,9)

In [23]:
X.shape

(22950, 15, 9)

In [25]:
# Use argmax to transform y

y =  np.argmax(y, axis = 1)
y

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [50]:
# Reshape X as a 3D object
X = X.reshape(-1,15,9)

In [27]:
# Check shape
y.shape

(22950,)

## 4. Split Data

In [29]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state = 42)

In [31]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(17212, 15, 9) (17212,)
(5738, 15, 9) (5738,)


## 5. Bayesian Hyperparameter Optimisation

In [34]:
timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = 15 # Number of weather stations
# Make scorer accuracy
score_acc = make_scorer(accuracy_score)

In [36]:
# Create function

def bay_area(neurons, activation, kernel, optimizer, learning_rate, batch_size, epochs,
              layers1, layers2, normalization, dropout, dropout_rate): 
    optimizerL = ['SGD', 'Adam', 'RMSprop', 'Adadelta', 'Adagrad', 'Adamax', 'Nadam', 'Ftrl','SGD']
    #optimizerD= {'Adam':Adam(lr=learning_rate), 'SGD':SGD(lr=learning_rate),
                 #'RMSprop':RMSprop(lr=learning_rate), 'Adadelta':Adadelta(lr=learning_rate),
                 #'Adagrad':Adagrad(lr=learning_rate), 'Adamax':Adamax(lr=learning_rate),
                 #'Nadam':Nadam(lr=learning_rate), 'Ftrl':Ftrl(lr=learning_rate)}
    activationL = ['relu', 'sigmoid', 'softplus', 'softsign', 'tanh', 'selu',
                   'elu', 'exponential', LeakyReLU,'relu']
    
    neurons = round(neurons)
    kernel = round(kernel)
    activation = activationL[round(activation)]  #optimizerD[optimizerL[round(optimizer)]]
    optimizer = optimizerL[round(optimizer)]
    batch_size = round(batch_size)
    
    epochs = round(epochs)
    layers1 = round(layers1)
    layers2 = round(layers2)
    
    def cnn_model():
        model = Sequential()
        model.add(Conv1D(neurons, kernel_size=kernel,activation=activation, input_shape=(timesteps, input_dim)))
        #model.add(Conv1D(32, kernel_size=1,activation='relu', input_shape=(timesteps, input_dim)))
        
        if normalization > 0.5:
            model.add(BatchNormalization())
        for i in range(layers1):
            model.add(Dense(neurons, activation=activation)) #(neurons, activation=activation))
        if dropout > 0.5:
            model.add(Dropout(dropout_rate, seed=123))
        for i in range(layers2):
            model.add(Dense(neurons, activation=activation))
        model.add(MaxPooling1D())
        model.add(Flatten())
        model.add(Dense(n_classes, activation='softmax')) #sigmoid softmax
        #model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy']) #categorical_crossentropy
        model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy']) #categorical_crossentropy
        return model
    es = EarlyStopping(monitor='accuracy', mode='max', verbose=2, patience=20)
    nn = KerasClassifier(build_fn=cnn_model, epochs=epochs, batch_size=batch_size, verbose=2)
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    score = cross_val_score(nn, X_train, y_train, scoring=score_acc, cv=kfold, params={'callbacks':[es]}).mean()
    return score

In [38]:
start = time.time()
params ={
    'neurons': (10, 100),
    'kernel': (1, 3),
    'activation':(0, 9), 
    'optimizer':(0,7),
    'learning_rate':(0.01, 1),
    'batch_size': (200, 1000), 
    'epochs':(20, 50),
    'layers1':(1,3),
    'layers2':(1,3),
    'normalization':(0,1),
    'dropout':(0,1),
    'dropout_rate':(0,0.3)
}
# Run Bayesian Optimization
nn_opt = BayesianOptimization(bay_area, params, random_state=42)
nn_opt.maximize(init_points=15, n_iter=4) 
print('Search took %s minutes' % ((time.time() - start)/60))

|   iter    |  target   |  neurons  |  kernel   | activa... | optimizer | learni... | batch_... |  epochs   |  layers1  |  layers2  | normal... |  dropout  | dropou... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------


C:\Users\daxma\anaconda3\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
C:\Users\daxma\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/22
43/43 - 3s - 65ms/step - accuracy: 0.6435 - loss: nan
Epoch 2/22
43/43 - 0s - 10ms/step - accuracy: 0.6440 - loss: nan
Epoch 3/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 4/22
43/43 - 0s - 8ms/step - accuracy: 0.6440 - loss: nan
Epoch 5/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 6/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 7/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 8/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 9/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 10/22
43/43 - 0s - 8ms/step - accuracy: 0.6440 - loss: nan
Epoch 11/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 12/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 13/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 14/22
43/43 - 1s - 12ms/step - accuracy: 0.6440 - loss: nan
Epoch 15/22
43/43 - 0s - 8ms/step - accuracy: 0.6440 - loss: nan
Epoch 16/22
43/43 - 0s - 9ms/st

C:\Users\daxma\anaconda3\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
C:\Users\daxma\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


43/43 - 3s - 65ms/step - accuracy: 0.6439 - loss: nan
Epoch 2/22
43/43 - 0s - 10ms/step - accuracy: 0.6440 - loss: nan
Epoch 3/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 4/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 5/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 6/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 7/22
43/43 - 0s - 10ms/step - accuracy: 0.6440 - loss: nan
Epoch 8/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 9/22
43/43 - 0s - 10ms/step - accuracy: 0.6440 - loss: nan
Epoch 10/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 11/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 12/22
43/43 - 0s - 10ms/step - accuracy: 0.6440 - loss: nan
Epoch 13/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 14/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 15/22
43/43 - 0s - 9ms/step - accuracy: 0.6440 - loss: nan
Epoch 16/22
43/43 - 0s - 9ms/step - accu

...

C:\Users\daxma\anaconda3\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
C:\Users\daxma\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


37/37 - 1s - 35ms/step - accuracy: 0.5816 - loss: 1.4305
Epoch 2/39
37/37 - 0s - 7ms/step - accuracy: 0.6539 - loss: 1.0848
Epoch 3/39
37/37 - 0s - 6ms/step - accuracy: 0.6550 - loss: 1.0303
Epoch 4/39
37/37 - 0s - 7ms/step - accuracy: 0.6643 - loss: 0.9971
Epoch 5/39
37/37 - 0s - 10ms/step - accuracy: 0.6731 - loss: 0.9749
Epoch 6/39
37/37 - 0s - 7ms/step - accuracy: 0.6806 - loss: 0.9521
Epoch 7/39
37/37 - 0s - 8ms/step - accuracy: 0.6794 - loss: 0.9365
Epoch 8/39
37/37 - 0s - 7ms/step - accuracy: 0.6854 - loss: 0.9207
Epoch 9/39
37/37 - 0s - 8ms/step - accuracy: 0.6901 - loss: 0.9126
Epoch 10/39
37/37 - 0s - 6ms/step - accuracy: 0.6946 - loss: 0.8967
Epoch 11/39
37/37 - 0s - 7ms/step - accuracy: 0.7005 - loss: 0.8859
Epoch 12/39
37/37 - 0s - 8ms/step - accuracy: 0.7033 - loss: 0.8752
Epoch 13/39
37/37 - 0s - 5ms/step - accuracy: 0.7072 - loss: 0.8664
Epoch 14/39
37/37 - 0s - 6ms/step - accuracy: 0.7081 - loss: 0.8534
Epoch 15/39
37/37 - 0s - 11ms/step - accuracy: 0.7133 - loss: 0.84

C:\Users\daxma\anaconda3\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
C:\Users\daxma\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


37/37 - 1s - 38ms/step - accuracy: 0.6078 - loss: 1.4229
Epoch 2/39
37/37 - 0s - 10ms/step - accuracy: 0.6511 - loss: 1.0932
Epoch 3/39
37/37 - 0s - 10ms/step - accuracy: 0.6599 - loss: 1.0441
Epoch 4/39
37/37 - 0s - 6ms/step - accuracy: 0.6662 - loss: 1.0099
Epoch 5/39
37/37 - 0s - 8ms/step - accuracy: 0.6755 - loss: 0.9815
Epoch 6/39
37/37 - 0s - 8ms/step - accuracy: 0.6801 - loss: 0.9617
Epoch 7/39
37/37 - 0s - 6ms/step - accuracy: 0.6801 - loss: 0.9463
Epoch 8/39
37/37 - 0s - 7ms/step - accuracy: 0.6866 - loss: 0.9288
Epoch 9/39
37/37 - 0s - 9ms/step - accuracy: 0.6880 - loss: 0.9182
Epoch 10/39
37/37 - 0s - 5ms/step - accuracy: 0.6914 - loss: 0.9058
Epoch 11/39
37/37 - 0s - 8ms/step - accuracy: 0.6935 - loss: 0.8963
Epoch 12/39
37/37 - 0s - 6ms/step - accuracy: 0.6997 - loss: 0.8841
Epoch 13/39
37/37 - 0s - 6ms/step - accuracy: 0.6996 - loss: 0.8794
Epoch 14/39
37/37 - 0s - 8ms/step - accuracy: 0.7029 - loss: 0.8681
Epoch 15/39
37/37 - 0s - 8ms/step - accuracy: 0.6991 - loss: 0.86

ValueError: Input y contains NaN.

In [44]:
optimum = nn_opt.max['params']
learning_rate = optimum['learning_rate']

activationL = ['relu', 'sigmoid', 'softplus', 'softsign', 'tanh', 'selu', 'elu', 'exponential', LeakyReLU, 'relu']
optimum['activation'] = activationL[round(optimum['activation'])]

optimum['batch_size'] = round(optimum['batch_size'])
optimum['epochs'] = round(optimum['epochs'])
optimum['layers1'] = round(optimum['layers1'])
optimum['layers2'] = round(optimum['layers2'])
optimum['neurons'] = round(optimum['neurons'])

optimizerL = ['Adam', 'SGD', 'RMSprop', 'Adadelta', 'Adagrad', 'Adamax', 'Nadam', 'Ftrl', 'Adam']
optimizerD = {
    'Adam': Adam(learning_rate=learning_rate),
    'SGD': SGD(learning_rate=learning_rate),
    'RMSprop': RMSprop(learning_rate=learning_rate),
    'Adadelta': Adadelta(learning_rate=learning_rate),
    'Adagrad': Adagrad(learning_rate=learning_rate),
    'Adamax': Adamax(learning_rate=learning_rate),
    'Nadam': Nadam(learning_rate=learning_rate),
    'Ftrl': Ftrl(learning_rate=learning_rate)
}
optimum['optimizer'] = optimizerD[optimizerL[round(optimum['optimizer'])]]
optimum

{'neurons': 37,
 'kernel': 1.1953442280127677,
 'activation': 'elu',
 'optimizer': <keras.src.optimizers.adadelta.Adadelta at 0x2438c7f93a0>,
 'learning_rate': 0.13081785249633104,
 'batch_size': 596,
 'epochs': 21,
 'layers1': 3,
 'layers2': 2,
 'normalization': 0.662522284353982,
 'dropout': 0.31171107608941095,
 'dropout_rate': 0.15602040635334324}

## 5. CNN with Optimised Hyperparamters

In [66]:
# Set the model with optimized hyperparameters

epochs = 21
batch_size = 256

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = 15

layers1 = 3
layers2 = 2
activation = 'elu'
kernel = int(round(1.1953442280127677))  # Rounded kernel size for Conv1D
neurons = 37
normalization = 0.662522284353982
dropout = 0.31171107608941095
dropout_rate = 0.15602040635334324
optimizer = Adadelta(learning_rate=0.130817852496331045)  # Instantiate RMSprop with learning rate

model = Sequential()
model.add(Conv1D(neurons, kernel_size=kernel, activation=activation, input_shape=(timesteps, input_dim)))

if normalization > 0.5:
    model.add(BatchNormalization())

for i in range(layers1):
    model.add(Dense(neurons, activation=activation))

if dropout > 0.5:
    model.add(Dropout(dropout_rate))

for i in range(layers2):
    model.add(Dense(neurons, activation=activation))

model.add(MaxPooling1D())
model.add(Flatten())
model.add(Dense(n_classes, activation='softmax')) 

model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

C:\Users\daxma\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [68]:
model.summary()

Model: "sequential_76"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_76 (Conv1D)              │ (None, 15, 37)         │           370 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_41          │ (None, 15, 37)         │           148 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_391 (Dense)               │ (None, 15, 37)         │         1,406 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_392 (Dense)               │ (None, 15, 37)         │         1,406 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_393 (Dense)               │ (None, 15, 37)         │         1,406 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_394 (Dense)               │ (None, 15, 37)         │         1,406 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_395 (Dense)               │ (None, 15, 37)         │         1,406 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_76 (MaxPooling1D) │ (None, 7, 37)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_76 (Flatten)            │ (None, 259)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_396 (Dense)               │ (None, 15)             │         3,900 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,448 (44.72 KB)

 Trainable params: 11,374 (44.43 KB)

 Non-trainable params: 74 (296.00 B)

In [69]:
# Put the y_test set back into a one-hot configuration

y_train_one_hot = to_categorical(y_train, num_classes=15)

In [70]:
# Check shapes

print(f'X_train shape: {X_train.shape}')
print(f'y_train_one_hot shape: {y_train_one_hot.shape}')

X_train shape: (17212, 15, 9)
y_train_one_hot shape: (17212, 15)


In [71]:
# Compile the model with categorical_crossentropy

model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

In [72]:
# Fit the model to the data

model.fit(X_train, y_train_one_hot, batch_size=batch_size, epochs=epochs, verbose=2)

Epoch 1/21
68/68 - 4s - 59ms/step - accuracy: 0.5087 - loss: 1.7003
Epoch 2/21
68/68 - 1s - 11ms/step - accuracy: 0.6692 - loss: 1.0236
Epoch 3/21
68/68 - 1s - 10ms/step - accuracy: 0.6904 - loss: 0.9180
Epoch 4/21
68/68 - 1s - 9ms/step - accuracy: 0.7130 - loss: 0.8467
Epoch 5/21
68/68 - 1s - 10ms/step - accuracy: 0.7293 - loss: 0.7976
Epoch 6/21
68/68 - 1s - 11ms/step - accuracy: 0.7392 - loss: 0.7648
Epoch 7/21
68/68 - 1s - 10ms/step - accuracy: 0.7471 - loss: 0.7412
Epoch 8/21
68/68 - 1s - 8ms/step - accuracy: 0.7543 - loss: 0.7204
Epoch 9/21
68/68 - 1s - 9ms/step - accuracy: 0.7585 - loss: 0.7033
Epoch 10/21
68/68 - 1s - 10ms/step - accuracy: 0.7637 - loss: 0.6874
Epoch 11/21
68/68 - 1s - 13ms/step - accuracy: 0.7664 - loss: 0.6735
Epoch 12/21
68/68 - 1s - 11ms/step - accuracy: 0.7705 - loss: 0.6597
Epoch 13/21
68/68 - 1s - 11ms/step - accuracy: 0.7746 - loss: 0.6465
Epoch 14/21
68/68 - 1s - 9ms/step - accuracy: 0.7788 - loss: 0.6341
Epoch 15/21
68/68 - 1s - 12ms/step - accuracy: 

## 7. Confusion Matrix

In [74]:
# Define list of stations names

stations = {
0: 'BASEL',
1: 'BELGRADE',
2: 'BUDAPEST',
3: 'DEBILT',
4: 'DUSSELDORF',
5: 'HEATHROW',
6: 'KASSEL',
7: 'LJUBLJANA',
8: 'MAASTRICHT',
9: 'MADRID',
10: 'MUNCHENB',
11: 'OSLO',
12: 'SONNBLICK',
13: 'STOCKHOLM',
14: 'VALENTIA'
}

In [75]:
def confusion_matrix(y_true, y_pred, stations):
    # Check if y_true and y_pred are one-hot encoded or already class indices
    if y_true.ndim == 1:
        y_true_labels = y_true
    else:
        y_true_labels = np.argmax(y_true, axis=1)
    
    if y_pred.ndim == 1:
        y_pred_labels = y_pred
    else:
        y_pred_labels = np.argmax(y_pred, axis=1)
        
    # Map numeric labels to activity names
    y_true_series = pd.Series([stations[y] for y in y_true_labels])
    y_pred_series = pd.Series([stations[y] for y in y_pred_labels])
    
    return pd.crosstab(y_true_series, y_pred_series, rownames=['True'], colnames=['Pred'])

In [76]:
y_pred = model.predict(X_test)

180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [77]:
# Evaluate

print(confusion_matrix(y_test, y_pred, stations))

Pred        BASEL  BELGRADE  BUDAPEST  DEBILT  HEATHROW  LJUBLJANA  MADRID
True                                                                      
BASEL        3391       171        42       6        14          1      57
BELGRADE      244       796        17       1         4          0      30
BUDAPEST       40        54        91       3         5          3      18
DEBILT         21         8        18      22        10          1       2
DUSSELDORF      9         2         6       0         9          0       3
HEATHROW       22         2         5       3        33          0      17
KASSEL          6         2         2       0         0          1       0
LJUBLJANA      19         8         4       0         1         11      18
MAASTRICHT      5         0         1       0         1          0       2
MADRID         54        28        16       0        13          1     346
MUNCHENB        8         0         0       0         0          0       0
OSLO            2        